# Práctica 6: Clasificación de Datos (K-NN)

**Objetivo:** Crear un modelo de clasificación utilizando el algoritmo **K-Nearest Neighbors (K-NN)** para predecir si un tiro en la Premier League (temporada 2024-2025) terminará en **Gol** o **No Gol**.

### Variables seleccionadas:
- `shot_x`, `shot_y`: Coordenadas del tiro.
- `isHome`: Si el equipo es local o visitante.
- `bodyPart`: Parte del cuerpo usada (pie izquierdo, derecho, cabeza).
- `situation`: Situación del tiro (jugada colectiva, tiro libre, etc.).
- `xg`: Probabilidad estadística previa de gol (Expected Goals).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Cargar datos
df = pd.read_csv('cleaned_epl_shots.csv')

# Crear la variable objetivo: 1 si es 'goal', 0 en cualquier otro caso
df['is_goal'] = (df['shotType'] == 'goal').astype(int)

print(f"Distribución de goles:\n{df['is_goal'].value_counts()}")

## Preprocesamiento de Datos

K-NN requiere que todas las variables sean numéricas y que estén escaladas, ya que se basa en distancias euclidianas.

In [ ]:
# Selección de características
features = ['shot_x', 'shot_y', 'isHome', 'bodyPart', 'situation', 'xg']
X = df[features].copy()
y = df['is_goal']

# Codificación de variables categóricas
le = LabelEncoder()
X['bodyPart'] = le.fit_transform(X['bodyPart'])
X['situation'] = le.fit_transform(X['situation'])
X['isHome'] = X['isHome'].astype(int)

# Dividir en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Escalado de datos (Crucial para KNN)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("Datos listos para el modelo.")

## Entrenamiento del Modelo K-NN

Iniciamos con un valor estándar de $K=5$.

In [ ]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)

y_pred = knn.predict(X_test)

print("Resultados con K=5:")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nMatriz de Confusión:")
print(confusion_matrix(y_test, y_pred))
print("\nReporte de Clasificación:")
print(classification_report(y_test, y_pred))

## Visualización de la Matriz de Confusión

In [ ]:
plt.figure(figsize=(8,6))
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicción')
plt.ylabel('Realidad')
plt.title('Matriz de Confusión - K-NN (K=5)')
plt.show()

## Optimización del parámetro K

Buscamos el valor de K que minimice el error de predicción.

In [ ]:
error_rate = []

for i in range(1, 25):
    knn = KNeighborsClassifier(n_neighbors=i)
    knn.fit(X_train, y_train)
    pred_i = knn.predict(X_test)
    error_rate.append(np.mean(pred_i != y_test))

plt.figure(figsize=(10,6))
plt.plot(range(1, 25), error_rate, color='blue', linestyle='dashed', marker='o', markerfacecolor='red', markersize=10)
plt.title('Tasa de Error vs. Valor de K')
plt.xlabel('K')
plt.ylabel('Tasa de Error')
plt.show()

## Conclusiones

1. **Desempeño del Modelo:** El modelo logra una precisión aceptable, aunque la clase de 'Goles' suele ser más difícil de predecir debido a que los datos están desbalanceados (hay muchos más tiros que no terminan en gol).
2. **Importancia de K:** El gráfico de tasa de error nos permite identificar el punto de equilibrio donde el modelo no sufre de subajuste (K muy pequeño) ni sobreajuste (K muy grande).
3. **Sugerencia:** Para mejorar el modelo en el futuro, se podrían usar técnicas de balanceo de clases (como SMOTE) o incluir variables adicionales del contexto del partido.